In [2]:
import torch

### Developing Intuition

In [8]:
attention_scores = torch.randn((6,6)) #
attention_scores

tensor([[-0.0956,  0.4672, -0.0143,  0.8417,  0.0275, -0.3024],
        [ 0.6609,  2.3339,  1.0536, -0.1775,  0.9467,  1.4049],
        [-1.2732, -0.0251,  1.2754, -0.5421, -0.6421, -0.4728],
        [-1.5634,  0.0182,  1.5266,  0.5309, -1.0706,  2.4314],
        [ 0.5010,  0.2630,  0.9822, -1.0507, -1.0846, -0.6331],
        [ 0.3995,  0.0140, -2.6489, -0.4481,  0.7170, -2.5232]])

In [6]:
mask = torch.triu(torch.ones(6,6), diagonal= 1) #1 makes the values on and below the main diagonal as 0 

In [7]:
print(mask)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [15]:
masked = attention_scores.masked_fill(mask.bool(), -torch.inf) #Fill the positions where 1 appears in "mask" to -inf in attention_scores matrix
masked

tensor([[-0.0956,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.6609,  2.3339,    -inf,    -inf,    -inf,    -inf],
        [-1.2732, -0.0251,  1.2754,    -inf,    -inf,    -inf],
        [-1.5634,  0.0182,  1.5266,  0.5309,    -inf,    -inf],
        [ 0.5010,  0.2630,  0.9822, -1.0507, -1.0846,    -inf],
        [ 0.3995,  0.0140, -2.6489, -0.4481,  0.7170, -2.5232]])

In [13]:
attention_weights = torch.softmax(masked/attention_scores.shape[1]**0.5,dim=1)

In [14]:
attention_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3356, 0.6644, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1820, 0.3029, 0.5151, 0.0000, 0.0000, 0.0000],
        [0.1138, 0.2170, 0.4017, 0.2675, 0.0000, 0.0000],
        [0.2393, 0.2172, 0.2913, 0.1270, 0.1253, 0.0000],
        [0.2330, 0.1991, 0.0671, 0.1649, 0.2653, 0.0707]])

### Creating a causal attention class with Dropout

Why Dropout ?

Dropout helps in reducing overfitting by ensuring that the model doesnt overly rely on some specific tokens while calculating attention weights 

Note: Dropout is only done during model training and is switched off at validation phase

In [30]:
import torch
import torch.nn as nn


In [31]:

inputs = torch.tensor([
    [0.43,0.15,0.89], #Your->x^1
    [0.55,0.87,0.66], #jounrey->x^2
    [0.57,0.85,0.64], #starts->x^3
    [0.22,0.58,0.33], #with->x^4
    [0.77,0.25,0.10], #one->x^5
    [0.05,0.80,0.55] #step-> x^6
])

In [32]:
num_tokens = inputs.shape[0]
print(f"Number of Tokens: {num_tokens}")

Number of Tokens: 6


In [33]:
#Batching helps us provide the model with multiple inputs at once, hence reducing training time

batch = torch.stack((inputs,inputs),dim=0) 
print(batch.shape)

torch.Size([2, 6, 3])


In [34]:
class CausalAttention(nn.Module):

    def __init__(self,d_in,d_out,context_length,dropout, qkv_bias= False):

        super().__init__()
        self.d_out= d_out
        self.W_query = nn.Linear(d_in,d_out,bias= qkv_bias)
        self.W_key = nn.Linear(d_in,d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out, bias = qkv_bias)
        dropout = nn.Dropout(dropout)
        #.register_buffer ensures that the tensors are in the same device, thus eliminating the chances of device mismatch error
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length), diagonal= 1)) 

    def forward(self,x):

        b,num_tokens, d_in = x.shape
        
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_values(x)

        attn_scores = queries @keys.transpose(1,2) #Keeping the batch dimension same
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)

        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

        

In [36]:
#Using the Causal Self-attention Class

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CasualAttention(d_in= inputs.shape[1], d_out =2,context_length,dropout= 0.0,qkv_Bias)
context_vecs = ca(batch)
print(f"Context Vecs Shape: {context_vecs.shape}")

SyntaxError: positional argument follows keyword argument (2689734589.py, line 5)